In [2]:
from google.colab import files
uploaded=files.upload()

Saving logins.txt to logins.txt


In [9]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Login Analysis").getOrCreate()

In [10]:
df=spark.read.text("logins.txt")
df.show()

+-----+
|value|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


In [11]:
df.withColumnRenamed("value","Name").show()

+-----+
| Name|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


In [13]:
print("Total logins : ",df.count())

Total logins :  26


In [19]:
df.select("value").distinct().show()

+-----+
|value|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



In [26]:
from pyspark.sql.functions import count
loginperuser=df.groupBy("value").count()
loginperuser.show()

+-----+-----+
|value|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



In [30]:
print("top 3 Active Users")
loginperuser.orderBy("count",ascending=False).show(3)

top 3 Active Users
+-----+-----+
|value|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


In [33]:
from pyspark.sql.functions import col
loginperuser.filter(col("count")>4).show()

+-----+-----+
|value|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



In [37]:
print("text file to dictionary")
dictform={row["value"]:row["count"] for row in loginperuser.collect()}
print(dictform)

text file to dictionary
{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


In [38]:
from google.colab import files
uploaded=files.upload()

Saving employees.csv to employees.csv


In [39]:
empdf=spark.read.csv("employees.csv",header=True,inferSchema=True)

In [43]:
print("Total Employees",empdf.count())

Total Employees 20


In [44]:
from pyspark.sql.functions import col
empdf.filter(col("department")=="IT").show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     5| Karan|        IT| 50000|   Mumbai|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    11| Anita|        IT| 65000|Bangalore|
|    13| Divya|        IT| 77000|Hyderabad|
|    15| Pooja|        IT| 69000|Bangalore|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [45]:
empdf.filter(col("salary")>75000).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



In [47]:
from pyspark.sql.functions import avg
empdf.select(avg("salary").alias("average salary")).show()

+--------------+
|average salary|
+--------------+
|       71450.0|
+--------------+



In [48]:
empdf.orderBy("salary",ascending=False).show(1)

+------+-----+----------+------+-----+
|emp_id| name|department|salary| city|
+------+-----+----------+------+-----+
|    20|Akash|   Finance| 91000|Delhi|
+------+-----+----------+------+-----+
only showing top 1 row


In [49]:
empdf.orderBy("salary",ascending=True).show(1)

+------+-----+----------+------+------+
|emp_id| name|department|salary|  city|
+------+-----+----------+------+------+
|     5|Karan|        IT| 50000|Mumbai|
+------+-----+----------+------+------+
only showing top 1 row


In [51]:
empdf.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



In [55]:
empdf.groupBy("department").avg("salary").show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



In [56]:
empdf.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+



In [57]:
empdf.orderBy(col("salary"),ascending=False).show(5)

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+
only showing top 5 rows


In [61]:
empdf.filter((col("city")=="Hyderabad")&(col("salary")>70000)).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [62]:
from google.colab import files
uploaded=files.upload()

Saving sales.csv to sales.csv


In [63]:
salesdf=spark.read.csv("sales.csv",header=True,inferSchema=True)

In [67]:
salesdf.withColumn("Revenue",col("quantity")*col("price")).show()

+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|Revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

In [68]:
from pyspark.sql.functions import sum

salesdf.withColumn("Revenue",col("quantity")*col("price")).select(sum("Revenue")).show()

+------------+
|sum(Revenue)|
+------------+
|      529500|
+------------+



In [72]:
salesdf.withColumn("Revenue",col("quantity")*col("price")).groupBy("product").sum("Revenue").show()

+--------+------------+
| product|sum(Revenue)|
+--------+------------+
|  Laptop|      450000|
|   Mouse|        7500|
|Keyboard|       12000|
| Monitor|       60000|
+--------+------------+



In [73]:
salesdf.groupBy("product").sum("quantity").show()

+--------+-------------+
| product|sum(quantity)|
+--------+-------------+
|  Laptop|            6|
|   Mouse|           15|
|Keyboard|            8|
| Monitor|            5|
+--------+-------------+



In [76]:
salesdf.groupBy("product").sum("quantity").orderBy(sum("quantity"),ascending=False).show(1)

+-------+-------------+
|product|sum(quantity)|
+-------+-------------+
|  Mouse|           15|
+-------+-------------+
only showing top 1 row


In [82]:
salesdf=salesdf.withColumn("Revenue",col("quantity")*col("price"))
salesdf.groupBy("emp_id").sum("Revenue").orderBy(sum("Revenue"),ascending=False).show(1)

+------+------------+
|emp_id|sum(Revenue)|
+------+------------+
|     1|      162000|
+------+------------+
only showing top 1 row


In [83]:
salesdf.select(avg("Revenue")).show()

+------------+
|avg(Revenue)|
+------------+
|     26475.0|
+------------+



In [84]:
from google.colab import files
uploaded=files.upload()


Saving orders.json to orders.json


In [97]:
ordersdf = spark.read.option("multiLine", "true").json("orders.json")
from pyspark.sql.functions import explode
ordersdf = ordersdf.select(explode("orders").alias("order")).select("order.*")


In [98]:
ordersdf.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [103]:
print("Total orders: ",ordersdf.count())

Total orders:  10


In [107]:
from pyspark.sql.functions import sum
ordersdf.select(sum("amount").alias("Total sales")).show()

+-----------+
|Total sales|
+-----------+
|     259500|
+-----------+



In [112]:
ordersdf.groupBy("customer").agg(sum("amount").alias("Total Revenue")).show()

+--------+-------------+
|customer|Total Revenue|
+--------+-------------+
|   Sneha|        76500|
|   Priya|        77000|
|   Rahul|        88000|
|   Arjun|         6000|
|   Karan|        12000|
+--------+-------------+



In [114]:
ordersdf.groupBy("customer").agg(sum("amount").alias("Total Revenue")).orderBy("Total Revenue",ascending =False).show(1)

+--------+-------------+
|customer|Total Revenue|
+--------+-------------+
|   Rahul|        88000|
+--------+-------------+
only showing top 1 row


In [115]:
ordersdf.groupBy("product").agg(sum("amount").alias("Total Revenue per product")).show()

+--------+-------------------------+
| product|Total Revenue per product|
+--------+-------------------------+
|  Laptop|                   225000|
|   Mouse|                     4500|
|Keyboard|                     6000|
| Monitor|                    24000|
+--------+-------------------------+



In [116]:
ordersdf.filter(col("city")=="Hyderabad").show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
|  1000|Hyderabad|   Rahul|       6|  Mouse|
|  2000|Hyderabad|   Priya|       9|  Mouse|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [117]:
ordersdf.filter(col("amount")>10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [119]:
ordersdf.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



In [128]:
salesdf = salesdf.withColumn("Revenue", col("quantity") * col("price"))
joindf=salesdf.join(empdf,on="emp_id",how="inner")
joindf.groupby("name").agg(sum("Revenue") .alias ("Total Revenue")).show()
joindf.groupby("department").agg(sum("Revenue") .alias ("Total Revenue")).orderBy("Total Revenue",ascending=False).show(1)
joindf.write.csv("final_sales_report.csv",header=True)

+------+-------------+
|  name|Total Revenue|
+------+-------------+
| Kunal|         1500|
| Sonal|        75000|
| Divya|        75000|
|  Ravi|         3000|
|Sanjay|         1500|
| Meera|        24000|
| Sneha|         1500|
| Priya|        75000|
|Vikram|        75000|
| Rahul|       162000|
| Anita|        12000|
| Manoj|         3000|
| Pooja|        12000|
| Arjun|         4000|
|  Amit|         2000|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+

+----------+-------------+
|department|Total Revenue|
+----------+-------------+
|        IT|       269500|
+----------+-------------+
only showing top 1 row
